# 🧬 Dark Manifold V11 — Boolean Essentiality + Neural Dynamics

## The V8-V10 Failure

We tried to **LEARN** gene essentiality from trajectory matching.
But trajectory matching doesn't encode essentiality — it's about **network topology**.

## V11 Philosophy: Separate Concerns

| Task | Method | Why |
|------|--------|-----|
| Trajectory prediction | Neural network | Learning works |
| Gene essentiality | Boolean logic | Topology, not dynamics |

## Boolean Essentiality Logic

A gene is **essential** if:
1. It's required for a reaction that...
2. Is the **ONLY** producer of an essential metabolite
3. OR is on the **critical path** to biomass

No learning needed — just graph traversal!

In [ ]:
#@title 1️⃣ Setup
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm
from collections import defaultdict, deque
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("\nUpload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()

# Parse genes from GPR rules
genes = set()
gene_to_rxns = {}
rxn_to_genes = defaultdict(set)

for j, gpr in enumerate(rxns_df['GPR rule'].tolist()):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)
        rxn_to_genes[j].add(g)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation'].lower()
    for i, name in enumerate(met_names):
        if name.lower() in equation:
            left = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[i, j] = -1 if name.lower() in left else +1

# Gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

# Key metabolite indices
def find_met_idx(keyword, exact=False):
    for i, n in enumerate(met_names):
        if exact:
            if keyword.lower() == n.lower():
                return i
        else:
            if keyword.lower() in n.lower():
                return i
    return None

# Find essential metabolites
atp_idx = find_met_idx('atp', exact=True) or find_met_idx('atp')
adp_idx = find_met_idx('adp', exact=True) or find_met_idx('adp')
amp_idx = find_met_idx('amp', exact=True) or find_met_idx('amp')
gtp_idx = find_met_idx('gtp', exact=True) or find_met_idx('gtp')
ctp_idx = find_met_idx('ctp', exact=True) or find_met_idx('ctp')
utp_idx = find_met_idx('utp', exact=True) or find_met_idx('utp')
nad_idx = find_met_idx('nad+') or find_met_idx('nad')
nadh_idx = find_met_idx('nadh')
nadp_idx = find_met_idx('nadp+')
nadph_idx = find_met_idx('nadph')
coa_idx = find_met_idx('coenzyme a') or find_met_idx('coa')
glucose_idx = find_met_idx('d-glucose')
pyruvate_idx = find_met_idx('pyruvate')
lactate_idx = find_met_idx('lactate')

ENV_VARS = ['glucose_ext', 'lactate_ext', 'oxygen', 'amino_acids', 'ph', 'temperature']
n_env = len(ENV_VARS)

print(f"Data: {n_genes} genes, {n_mets} metabolites, {n_rxns} reactions")
print(f"\nKey metabolites:")
print(f"  ATP: {atp_idx} ({met_names[atp_idx] if atp_idx else 'N/A'})")
print(f"  GTP: {gtp_idx} ({met_names[gtp_idx] if gtp_idx else 'N/A'})")
print(f"  NAD: {nad_idx} ({met_names[nad_idx] if nad_idx else 'N/A'})")

In [ ]:
#@title 3️⃣ Boolean Essentiality Analysis

class BooleanEssentialityAnalyzer:
    """
    V11 CORE: Determine gene essentiality using network topology.
    
    A gene is essential if knocking it out blocks ALL paths to
    an essential metabolite.
    """
    
    def __init__(self, S, G, genes, met_names, rxn_to_genes):
        self.S = S
        self.G = G
        self.genes = genes
        self.met_names = met_names
        self.rxn_to_genes = rxn_to_genes
        self.n_mets, self.n_rxns = S.shape
        self.n_genes = len(genes)
        
        # Build reaction maps
        self.rxn_produces = defaultdict(list)  # rxn -> [mets produced]
        self.rxn_consumes = defaultdict(list)  # rxn -> [mets consumed]
        self.met_produced_by = defaultdict(list)  # met -> [rxns that produce it]
        self.met_consumed_by = defaultdict(list)  # met -> [rxns that consume it]
        
        for rxn in range(self.n_rxns):
            for met in range(self.n_mets):
                if S[met, rxn] > 0:
                    self.rxn_produces[rxn].append(met)
                    self.met_produced_by[met].append(rxn)
                elif S[met, rxn] < 0:
                    self.rxn_consumes[rxn].append(met)
                    self.met_consumed_by[met].append(rxn)
        
        # Essential metabolites (required for life)
        self.essential_mets = self._find_essential_metabolites()
        
        print(f"Essential metabolites identified: {len(self.essential_mets)}")
    
    def _find_essential_metabolites(self):
        """Identify metabolites required for cell survival."""
        essential = set()
        
        # Energy currency
        for keyword in ['atp', 'gtp', 'ctp', 'utp']:
            for i, name in enumerate(self.met_names):
                if keyword == name.lower() or (keyword in name.lower() and 'd' + keyword not in name.lower()):
                    essential.add(i)
                    break
        
        # Redox carriers
        for keyword in ['nad+', 'nadh', 'nadp+', 'nadph', 'fad', 'fadh']:
            for i, name in enumerate(self.met_names):
                if keyword in name.lower():
                    essential.add(i)
                    break
        
        # Amino acids (for protein synthesis)
        amino_acids = ['alanine', 'arginine', 'asparagine', 'aspartate', 'cysteine',
                       'glutamate', 'glutamine', 'glycine', 'histidine', 'isoleucine',
                       'leucine', 'lysine', 'methionine', 'phenylalanine', 'proline',
                       'serine', 'threonine', 'tryptophan', 'tyrosine', 'valine']
        for aa in amino_acids:
            for i, name in enumerate(self.met_names):
                if aa in name.lower() and 'trna' not in name.lower():
                    essential.add(i)
                    break
        
        return essential
    
    def get_reactions_for_gene(self, gene_idx):
        """Get all reactions that require this gene."""
        return np.where(self.G[gene_idx] > 0)[0]
    
    def can_produce_metabolite(self, met_idx, blocked_rxns):
        """
        Check if metabolite can still be produced when some reactions are blocked.
        """
        producing_rxns = self.met_produced_by[met_idx]
        
        # If any producing reaction is NOT blocked, metabolite can be made
        for rxn in producing_rxns:
            if rxn not in blocked_rxns:
                return True
        
        return False
    
    def is_essential_simple(self, gene_idx):
        """
        Simple essentiality check:
        Gene is essential if blocking its reactions blocks ALL paths
        to ANY essential metabolite.
        """
        # Get reactions this gene controls
        blocked_rxns = set(self.get_reactions_for_gene(gene_idx))
        
        if len(blocked_rxns) == 0:
            return False, "No reactions controlled"
        
        # Check each essential metabolite
        for met_idx in self.essential_mets:
            if not self.can_produce_metabolite(met_idx, blocked_rxns):
                met_name = self.met_names[met_idx]
                return True, f"Blocks all production of {met_name}"
        
        return False, "Alternative pathways exist"
    
    def is_essential_strict(self, gene_idx):
        """
        Strict essentiality: Gene is essential if it's the ONLY gene
        for a reaction that's the ONLY producer of an essential metabolite.
        """
        controlled_rxns = self.get_reactions_for_gene(gene_idx)
        
        for rxn in controlled_rxns:
            # Is this gene the ONLY one for this reaction?
            genes_for_rxn = self.rxn_to_genes.get(rxn, set())
            if len(genes_for_rxn) > 1:
                continue  # Other genes can compensate
            
            # What does this reaction produce?
            products = self.rxn_produces[rxn]
            
            for met in products:
                if met in self.essential_mets:
                    # Is this the ONLY reaction producing this metabolite?
                    other_producers = [r for r in self.met_produced_by[met] if r != rxn]
                    if len(other_producers) == 0:
                        return True, f"Only producer of {self.met_names[met]}"
        
        return False, "Redundant pathways exist"
    
    def compute_all_essentiality(self, method='simple'):
        """
        Compute essentiality for all genes.
        """
        results = []
        
        for i, gene in enumerate(self.genes):
            if method == 'simple':
                is_ess, reason = self.is_essential_simple(i)
            else:
                is_ess, reason = self.is_essential_strict(i)
            
            rxns_controlled = len(self.get_reactions_for_gene(i))
            
            results.append({
                'gene': gene,
                'gene_idx': i,
                'is_essential': is_ess,
                'reason': reason,
                'rxns_controlled': rxns_controlled,
            })
        
        return results


# Run analysis
analyzer = BooleanEssentialityAnalyzer(S, G, genes, met_names, rxn_to_genes)

print("\n" + "="*60)
print("BOOLEAN ESSENTIALITY ANALYSIS")
print("="*60)

# Simple method
results_simple = analyzer.compute_all_essentiality(method='simple')
essential_simple = [r for r in results_simple if r['is_essential']]
print(f"\nSimple method: {len(essential_simple)}/{n_genes} essential ({100*len(essential_simple)/n_genes:.1f}%)")

# Strict method
results_strict = analyzer.compute_all_essentiality(method='strict')
essential_strict = [r for r in results_strict if r['is_essential']]
print(f"Strict method: {len(essential_strict)}/{n_genes} essential ({100*len(essential_strict)/n_genes:.1f}%)")

# Show essential genes
print(f"\nEssential genes (simple method):")
for r in essential_simple[:15]:
    print(f"  {r['gene']}: {r['reason']} (controls {r['rxns_controlled']} rxns)")
if len(essential_simple) > 15:
    print(f"  ... and {len(essential_simple) - 15} more")

In [ ]:
#@title 4️⃣ Enhanced Essentiality: Pathway Reachability

class PathwayReachabilityAnalyzer(BooleanEssentialityAnalyzer):
    """
    Enhanced essentiality using pathway reachability analysis.
    
    A gene is essential if knocking it out makes an essential
    metabolite UNREACHABLE from the available nutrients.
    """
    
    def __init__(self, S, G, genes, met_names, rxn_to_genes):
        super().__init__(S, G, genes, met_names, rxn_to_genes)
        
        # Define nutrients (always available)
        self.nutrients = self._find_nutrients()
        print(f"Nutrients identified: {len(self.nutrients)}")
    
    def _find_nutrients(self):
        """Identify metabolites available from the environment."""
        nutrients = set()
        
        # Common nutrients
        nutrient_keywords = ['glucose', 'amino acid', 'phosphate', 'ammonia', 
                             'water', 'h2o', 'oxygen', 'o2', 'co2', 'carbon dioxide']
        
        for keyword in nutrient_keywords:
            for i, name in enumerate(self.met_names):
                if keyword in name.lower():
                    nutrients.add(i)
        
        # Also: metabolites with no consumers (likely external)
        for i in range(self.n_mets):
            if len(self.met_consumed_by[i]) == 0 and len(self.met_produced_by[i]) > 0:
                nutrients.add(i)
        
        return nutrients
    
    def compute_reachable_metabolites(self, blocked_genes=None):
        """
        BFS to find all metabolites reachable from nutrients.
        
        A metabolite is reachable if:
        1. It's a nutrient, OR
        2. There's an active reaction that produces it from reachable substrates
        """
        if blocked_genes is None:
            blocked_genes = set()
        
        # Find blocked reactions
        blocked_rxns = set()
        for gene_idx in blocked_genes:
            for rxn in self.get_reactions_for_gene(gene_idx):
                blocked_rxns.add(rxn)
        
        # BFS from nutrients
        reachable = set(self.nutrients)
        queue = deque(self.nutrients)
        
        while queue:
            met = queue.popleft()
            
            # Find reactions that consume this metabolite
            for rxn in self.met_consumed_by[met]:
                if rxn in blocked_rxns:
                    continue
                
                # Check if all substrates are reachable
                substrates = self.rxn_consumes[rxn]
                if all(s in reachable for s in substrates):
                    # Products become reachable
                    for product in self.rxn_produces[rxn]:
                        if product not in reachable:
                            reachable.add(product)
                            queue.append(product)
        
        return reachable
    
    def is_essential_reachability(self, gene_idx):
        """
        Gene is essential if its knockout makes an essential
        metabolite unreachable.
        """
        # Baseline: what's reachable with all genes?
        baseline_reachable = self.compute_reachable_metabolites(blocked_genes=set())
        
        # With knockout: what's reachable?
        ko_reachable = self.compute_reachable_metabolites(blocked_genes={gene_idx})
        
        # Check essential metabolites
        for met_idx in self.essential_mets:
            if met_idx in baseline_reachable and met_idx not in ko_reachable:
                return True, f"Makes {self.met_names[met_idx]} unreachable"
        
        return False, "All essential metabolites still reachable"
    
    def compute_all_essentiality_reachability(self):
        """Compute essentiality using reachability method."""
        results = []
        
        for i, gene in enumerate(tqdm(self.genes, desc="Analyzing genes")):
            is_ess, reason = self.is_essential_reachability(i)
            rxns_controlled = len(self.get_reactions_for_gene(i))
            
            results.append({
                'gene': gene,
                'gene_idx': i,
                'is_essential': is_ess,
                'reason': reason,
                'rxns_controlled': rxns_controlled,
            })
        
        return results


# Run enhanced analysis
print("\n" + "="*60)
print("PATHWAY REACHABILITY ANALYSIS")
print("="*60)

pathway_analyzer = PathwayReachabilityAnalyzer(S, G, genes, met_names, rxn_to_genes)

# Baseline reachability
baseline = pathway_analyzer.compute_reachable_metabolites()
print(f"\nBaseline reachable metabolites: {len(baseline)}/{n_mets}")

# Essential metabolites reachable?
essential_reachable = sum(1 for m in pathway_analyzer.essential_mets if m in baseline)
print(f"Essential metabolites reachable: {essential_reachable}/{len(pathway_analyzer.essential_mets)}")

# Full reachability analysis
results_reach = pathway_analyzer.compute_all_essentiality_reachability()
essential_reach = [r for r in results_reach if r['is_essential']]
print(f"\nReachability method: {len(essential_reach)}/{n_genes} essential ({100*len(essential_reach)/n_genes:.1f}%)")

if essential_reach:
    print(f"\nEssential genes (reachability):")
    for r in essential_reach[:15]:
        print(f"  {r['gene']}: {r['reason']}")

In [ ]:
#@title 5️⃣ Flux-Based Essentiality (Mini-FBA)

class FluxEssentialityAnalyzer:
    """
    Simplified Flux Balance Analysis to determine essentiality.
    
    A gene is essential if its knockout reduces maximum ATP production to zero.
    """
    
    def __init__(self, S, G, genes, atp_idx):
        self.S = S
        self.G = G
        self.genes = genes
        self.atp_idx = atp_idx
        self.n_mets, self.n_rxns = S.shape
        self.n_genes = len(genes)
    
    def compute_atp_production(self, blocked_genes=None, max_iter=100):
        """
        Iterative flux computation to estimate ATP production capacity.
        
        This is a simplified proxy for FBA without scipy.
        """
        if blocked_genes is None:
            blocked_genes = set()
        
        # Reaction mask (0 if blocked, 1 if active)
        rxn_mask = np.ones(self.n_rxns)
        for gene_idx in blocked_genes:
            rxn_indices = np.where(self.G[gene_idx] > 0)[0]
            rxn_mask[rxn_indices] = 0
        
        # Simple flux estimation:
        # Start with equal flux, then scale by capacity
        flux = np.ones(self.n_rxns) * rxn_mask
        
        # Iteratively balance
        for _ in range(max_iter):
            # Compute net production
            net_prod = self.S @ flux
            
            # Reduce flux for reactions consuming scarce metabolites
            for rxn in range(self.n_rxns):
                if rxn_mask[rxn] == 0:
                    continue
                
                # Check substrate availability
                substrates = np.where(self.S[:, rxn] < 0)[0]
                if len(substrates) > 0:
                    min_availability = min(net_prod[s] + 10 for s in substrates)  # +10 baseline
                    if min_availability < 0:
                        flux[rxn] *= 0.9  # Reduce flux
            
            flux = np.clip(flux, 0, 10)
        
        # ATP production = sum of flux into ATP
        atp_production = 0
        for rxn in range(self.n_rxns):
            if self.S[self.atp_idx, rxn] > 0:  # Produces ATP
                atp_production += flux[rxn] * self.S[self.atp_idx, rxn]
        
        return atp_production
    
    def compute_all_essentiality(self):
        """Compute essentiality based on ATP production capacity."""
        
        # Baseline ATP production
        baseline_atp = self.compute_atp_production(blocked_genes=set())
        print(f"Baseline ATP production: {baseline_atp:.2f}")
        
        results = []
        for i, gene in enumerate(tqdm(self.genes, desc="Flux analysis")):
            ko_atp = self.compute_atp_production(blocked_genes={i})
            atp_ratio = ko_atp / (baseline_atp + 1e-6)
            
            results.append({
                'gene': gene,
                'gene_idx': i,
                'atp_production': ko_atp,
                'atp_ratio': atp_ratio,
                'is_essential': atp_ratio < 0.1,  # <10% ATP = essential
            })
        
        return results


print("\n" + "="*60)
print("FLUX-BASED ESSENTIALITY ANALYSIS")
print("="*60)

if atp_idx is not None:
    flux_analyzer = FluxEssentialityAnalyzer(S, G, genes, atp_idx)
    results_flux = flux_analyzer.compute_all_essentiality()
    
    essential_flux = [r for r in results_flux if r['is_essential']]
    print(f"\nFlux method: {len(essential_flux)}/{n_genes} essential ({100*len(essential_flux)/n_genes:.1f}%)")
    
    # ATP ratio distribution
    atp_ratios = [r['atp_ratio'] for r in results_flux]
    print(f"ATP ratio range: [{min(atp_ratios):.3f}, {max(atp_ratios):.3f}]")
    print(f"ATP ratio std: {np.std(atp_ratios):.4f}")
else:
    print("ATP index not found, skipping flux analysis")
    results_flux = []

In [ ]:
#@title 6️⃣ Combine All Methods

print("="*60)
print("COMBINED ESSENTIALITY RESULTS")
print("="*60)

# Combine results
combined_results = []

for i, gene in enumerate(genes):
    simple_ess = results_simple[i]['is_essential'] if i < len(results_simple) else False
    strict_ess = results_strict[i]['is_essential'] if i < len(results_strict) else False
    reach_ess = results_reach[i]['is_essential'] if i < len(results_reach) else False
    flux_ess = results_flux[i]['is_essential'] if i < len(results_flux) else False
    flux_ratio = results_flux[i]['atp_ratio'] if i < len(results_flux) else 1.0
    
    # Consensus: essential if ANY method says so
    is_essential_any = simple_ess or strict_ess or reach_ess or flux_ess
    
    # Strict consensus: essential if MULTIPLE methods agree
    n_methods = sum([simple_ess, strict_ess, reach_ess, flux_ess])
    is_essential_consensus = n_methods >= 2
    
    combined_results.append({
        'gene': gene,
        'simple': simple_ess,
        'strict': strict_ess,
        'reachability': reach_ess,
        'flux': flux_ess,
        'flux_ratio': flux_ratio,
        'n_methods': n_methods,
        'essential_any': is_essential_any,
        'essential_consensus': is_essential_consensus,
    })

# Summary
ess_any = sum(1 for r in combined_results if r['essential_any'])
ess_consensus = sum(1 for r in combined_results if r['essential_consensus'])

print(f"\nMethod comparison:")
print(f"  Simple:       {len(essential_simple):3d} ({100*len(essential_simple)/n_genes:.1f}%)")
print(f"  Strict:       {len(essential_strict):3d} ({100*len(essential_strict)/n_genes:.1f}%)")
print(f"  Reachability: {len(essential_reach):3d} ({100*len(essential_reach)/n_genes:.1f}%)")
print(f"  Flux:         {len(essential_flux):3d} ({100*len(essential_flux)/n_genes:.1f}%)")
print(f"\n  Any method:   {ess_any:3d} ({100*ess_any/n_genes:.1f}%)")
print(f"  Consensus:    {ess_consensus:3d} ({100*ess_consensus/n_genes:.1f}%)")

print(f"\nExpected (Hutchison 2016): ~70% essential")

# Store for later use
boolean_essentiality = {r['gene']: r['essential_any'] for r in combined_results}

In [ ]:
#@title 7️⃣ V11 Neural Model (Trajectory Only)

class DarkManifoldV11(nn.Module):
    """
    V11: Neural network for TRAJECTORY prediction only.
    Essentiality is determined by Boolean logic, not learned.
    """
    
    def __init__(self, n_genes, n_mets, n_rxns, n_env, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        self.register_buffer('substrate_mask', (torch.tensor(S) < 0).float())
        
        # Gene regulation
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.1)
        
        # Environment sensing
        self.env_sensor = nn.Sequential(
            nn.Linear(n_env, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_genes),
            nn.Tanh(),
        )
        
        # Gene → Enzyme
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
            nn.Softplus(),
        )
        
        # Kinetics
        self.log_Km = nn.Parameter(torch.randn(n_rxns) * 0.5)
        
        # Hamiltonian
        self.hamiltonian_net = nn.Sequential(
            nn.Linear(n_mets + n_rxns, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    def forward(self, gene_expr, met_conc, env_state, dt=0.01):
        # Gene regulation
        reg_signal = torch.tanh(gene_expr @ self.W_reg.T)
        env_signal = self.env_sensor(env_state)
        regulated = gene_expr * (1.0 + 0.5 * reg_signal + 0.2 * env_signal)
        regulated = regulated.clamp(min=0.001)
        
        # Enzyme activity
        Vmax = self.gene_encoder(regulated)
        
        # Direct gene→flux via G matrix
        enzyme_level = regulated @ self.G  # (batch, n_rxns)
        enzyme_level = enzyme_level / (self.G.sum(dim=0).clamp(min=1))
        
        # Michaelis-Menten
        n_subs = self.substrate_mask.sum(dim=0).clamp(min=1)
        sub_conc = (met_conc @ self.substrate_mask) / n_subs + 0.001
        mm = sub_conc / (self.Km.unsqueeze(0) + sub_conc)
        
        # Flux
        flux = Vmax * mm * enzyme_level.clamp(0.01, 2.0)
        
        # Dynamics
        dM_dt = flux @ self.S.T
        met_next = (met_conc + dt * dM_dt).clamp(min=0.001)
        
        # Hamiltonian
        H = self.hamiltonian_net(torch.cat([met_conc, flux], dim=-1)).squeeze(-1)
        
        return {'met_next': met_next, 'flux': flux, 'hamiltonian': H}
    
    def rollout(self, gene_expr, met_init, env_init, n_steps, dt=0.01):
        met_traj = [met_init.clone()]
        met = met_init.clone()
        env = env_init.clone()
        
        for _ in range(n_steps):
            out = self.forward(gene_expr, met, env, dt)
            met = out['met_next']
            met_traj.append(met)
        
        return {'met_trajectory': torch.stack(met_traj, dim=1)}


model = DarkManifoldV11(
    n_genes=n_genes,
    n_mets=n_mets,
    n_rxns=n_rxns,
    n_env=n_env,
    S=S,
    G=G,
).to(device)

print(f"DarkManifoldV11: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
#@title 8️⃣ Quick Training

# Generator
class SimpleGenerator:
    def __init__(self, S, G, device):
        self.S = torch.tensor(S, dtype=torch.float32, device=device)
        self.G = torch.tensor(G, dtype=torch.float32, device=device)
        self.device = device
        self.n_mets = S.shape[0]
        self.n_genes = G.shape[0]
        self.substrate_mask = (self.S < 0).float()
    
    def simulate(self, n_steps=30, batch_size=8):
        gene_expr = torch.rand(batch_size, self.n_genes, device=self.device) * 1.5 + 0.5
        met = torch.rand(batch_size, self.n_mets, device=self.device) + 0.5
        met[:, atp_idx] = 4.0
        met[:, adp_idx] = 0.5
        
        traj = [met.clone()]
        for _ in range(n_steps):
            enzyme = gene_expr @ self.G
            n_subs = self.substrate_mask.sum(dim=0).clamp(min=1)
            sub_conc = (met @ self.substrate_mask) / n_subs + 0.001
            flux = enzyme * sub_conc / (1.0 + sub_conc) * 0.5
            met = (met + 0.01 * flux @ self.S.T).clamp(min=0.001)
            traj.append(met.clone())
        
        return {'gene_trajectory': gene_expr.unsqueeze(1).expand(-1, n_steps+1, -1),
                'met_trajectory': torch.stack(traj, dim=1)}

gen = SimpleGenerator(S, G, device)

# Training
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
losses = []

for epoch in tqdm(range(300), desc="Training V11"):
    model.train()
    
    with torch.no_grad():
        target = gen.simulate(n_steps=30, batch_size=8)
    
    gene = target['gene_trajectory'][:, 0]
    met_init = target['met_trajectory'][:, 0]
    env = torch.zeros(8, n_env, device=device)
    env[:, 0] = 10.0
    env[:, 4] = 7.0
    
    pred = model.rollout(gene, met_init.clone(), env, n_steps=30)
    loss = F.mse_loss(pred['met_trajectory'], target['met_trajectory'])
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
#@title 9️⃣ Final Results: Boolean + Neural

print("="*70)
print("DARK MANIFOLD V11 — FINAL RESULTS")
print("="*70)

# Trajectory evaluation
model.eval()
with torch.no_grad():
    test = gen.simulate(n_steps=50, batch_size=1)
    gene = test['gene_trajectory'][:, 0]
    met_init = test['met_trajectory'][:, 0]
    env = torch.zeros(1, n_env, device=device)
    env[:, 0] = 10.0
    env[:, 4] = 7.0
    
    pred = model.rollout(gene, met_init.clone(), env, n_steps=50)

true_traj = test['met_trajectory'][0].cpu().numpy()
pred_traj = pred['met_trajectory'][0].cpu().numpy()
corr = np.corrcoef(true_traj.flatten(), pred_traj.flatten())[0, 1]

print(f"\n[NEURAL NETWORK: Trajectory Prediction]")
print(f"  Correlation: {corr:.4f}")
print(f"  MSE: {np.mean((true_traj - pred_traj)**2):.6f}")

print(f"\n[BOOLEAN LOGIC: Gene Essentiality]")
print(f"  Simple method:       {len(essential_simple):3d} ({100*len(essential_simple)/n_genes:.1f}%)")
print(f"  Reachability method: {len(essential_reach):3d} ({100*len(essential_reach)/n_genes:.1f}%)")
print(f"  Flux method:         {len(essential_flux):3d} ({100*len(essential_flux)/n_genes:.1f}%)")
print(f"  Any method:          {ess_any:3d} ({100*ess_any/n_genes:.1f}%)")

# Plot
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Trajectories
met_indices = [atp_idx, adp_idx, glucose_idx or 0, lactate_idx or 1, 0, 1]
met_labels = ['ATP', 'ADP', 'Glucose', 'Lactate', 'Met 0', 'Met 1']

for ax, idx, label in zip(axes.flatten(), met_indices, met_labels):
    if idx is not None and idx < true_traj.shape[1]:
        ax.plot(true_traj[:, idx], 'b-', linewidth=2, label='True')
        ax.plot(pred_traj[:, idx], 'r--', linewidth=2, label='V11')
        ax.set_title(label)
        ax.legend()

plt.suptitle(f'V11 Trajectory (Corr={corr:.3f})', fontsize=14)
plt.tight_layout()
plt.savefig('trajectory_v11.png', dpi=150)
plt.show()

# Essentiality distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# By method
methods = ['Simple', 'Strict', 'Reachability', 'Flux']
counts = [len(essential_simple), len(essential_strict), len(essential_reach), len(essential_flux)]
axes[0].bar(methods, counts, color=['blue', 'green', 'orange', 'red'])
axes[0].axhline(y=0.7*n_genes, color='black', linestyle='--', label='Expected 70%')
axes[0].set_ylabel('Essential genes')
axes[0].set_title('Essential by Method')
axes[0].legend()

# Flux ATP ratios
if results_flux:
    atp_ratios = [r['atp_ratio'] for r in results_flux]
    axes[1].hist(atp_ratios, bins=30, edgecolor='black')
    axes[1].axvline(x=0.1, color='r', linestyle='--', label='Essential threshold')
    axes[1].set_xlabel('ATP ratio (KO/WT)')
    axes[1].set_title('Flux-Based ATP Ratio')
    axes[1].legend()

# Method agreement
n_methods_dist = [r['n_methods'] for r in combined_results]
axes[2].hist(n_methods_dist, bins=[0, 1, 2, 3, 4, 5], edgecolor='black', align='left')
axes[2].set_xlabel('Number of methods calling essential')
axes[2].set_ylabel('Genes')
axes[2].set_title('Method Agreement')

plt.tight_layout()
plt.savefig('essentiality_v11.png', dpi=150)
plt.show()

In [ ]:
#@title 🔟 Save Results

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'met_names': met_names,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'version': 'v11_boolean_neural',
    
    # Boolean essentiality results
    'essentiality': {
        'simple': results_simple,
        'strict': results_strict,
        'reachability': results_reach,
        'flux': results_flux,
        'combined': combined_results,
    },
    
    'metrics': {
        'trajectory_corr': float(corr),
        'n_essential_simple': len(essential_simple),
        'n_essential_reach': len(essential_reach),
        'n_essential_flux': len(essential_flux),
        'n_essential_any': ess_any,
        'n_essential_consensus': ess_consensus,
        'pct_essential_any': 100 * ess_any / n_genes,
    },
    
    'essential_mets': list(pathway_analyzer.essential_mets),
    'nutrients': list(pathway_analyzer.nutrients),
}

torch.save(save_dict, 'dark_manifold_v11.pt')

print("="*70)
print("V11 SUMMARY: BOOLEAN + NEURAL")
print("="*70)
print(f"""
TRAJECTORY PREDICTION (Neural Network):
  Correlation: {corr:.4f}

GENE ESSENTIALITY (Boolean Logic):
  Simple method:       {len(essential_simple):3d} genes ({100*len(essential_simple)/n_genes:.1f}%)
  Reachability method: {len(essential_reach):3d} genes ({100*len(essential_reach)/n_genes:.1f}%)
  Flux method:         {len(essential_flux):3d} genes ({100*len(essential_flux)/n_genes:.1f}%)
  
  Combined (any):      {ess_any:3d} genes ({100*ess_any/n_genes:.1f}%)
  Combined (consensus):{ess_consensus:3d} genes ({100*ess_consensus/n_genes:.1f}%)

COMPARISON TO PREVIOUS VERSIONS:
  V8-V10: 0% essential (trying to LEARN essentiality)
  V11:    {100*ess_any/n_genes:.0f}% essential (COMPUTE essentiality from topology)

KEY INSIGHT:
  Essentiality is a TOPOLOGICAL property, not a DYNAMICAL one.
  Use neural networks for dynamics, logic for topology.
""")

from google.colab import files
files.download('dark_manifold_v11.pt')
files.download('trajectory_v11.png')
files.download('essentiality_v11.png')